## Notebook40

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! python -m spacy download en_core_web_sm

In [ ]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs
from funs import DSText

import spacy

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [ ]:
docs = (
    pl.read_parquet(ub + "data/imdb5k_pca.parquet")
    .select(c.id, c.label, c.text)
    .rename({"id": "doc_id"})
    .sort(c.doc_id)
    .group_by(c.label, maintain_order=True)
    .head(100)
    .with_columns(
        text = c.text.str.replace_all("<br />", " ")
    )
)

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
anno = DSText.process(docs.select(c.doc_id, c.text), nlp)

### Questions

`docs` and `anno` carry the same 200 IMDb movie reviews as the last
notebook, 100 labeled `"positive"` and 100 labeled `"negative"`, with the
stray `<br />` HTML already replaced by a space before `anno` was built.
This notebook moves from single tokens and parts of speech to multi-word
phrases, named entities, and grammatical structure. Print results unless a
question says otherwise.

1. Build the bigram table the same way the chapter does: filter `anno` to
alphabetic tokens, shift `lemma` and `is_alpha` by `-1` within each `doc_id`
and `sid`, keep pairs where both tokens are alphabetic, and paste the two
lemmas together into a `bigram` column. Print the 15 most frequent bigrams.

As in the chapter, raw frequency mostly turns up function-word glue ("of
the", "in the", "to be") plus the two words for the thing being reviewed
("this movie", "the film"). Nothing here is specific to movie reviews; the
same bigrams would top the list for almost any English corpus.

2. Compute PMI for every bigram, following the chapter's chain: count each
word, count each bigram, join the two word counts onto the bigram table, and
combine them into `p_bigram`, `p_w1`, `p_w2`, and finally `pmi`. Filter to
`bigram_count >= 5` and print the 15 highest-PMI bigrams.

The chapter's PMI list for the Wikipedia author pages was almost entirely
proper nouns: book titles ("Biographia Literaria"), historical events
("Molotov-Ribbentrop Pact"), and names of people the authors wrote about.
This list has proper nouns too ("New York", "Cathy Lee"), but it also has
genre vocabulary no biography page would produce ("sci fi", "low budget",
"special effect", "production value", "support cast") and first-person
review phrasing ("my opinion", "my favorite", "apart from"). PMI measures how
strongly two words predict each other, and in a corpus of reviews some of
the most predictable pairs come from the genre's vocabulary and the
reviewer's voice.

3. Build the `entities` table the same way the chapter does: filter `anno`
to rows with a non-empty `ent_type`, mark the start of each entity with
`ent_iob == "B"`, take a cumulative sum of that flag within each `doc_id` to
get an `entity_id`, and group by `doc_id`, `entity_id`, and `ent_type` to
join the tokens back into `entity_text`. Then print the 15 most frequently
mentioned `PERSON` entities.

4. Group `entities` by `ent_type`, count, and plot the result as a bar chart
with `geom_col()` and `coord_flip()`, matching the chapter's own entity-type
plot.

`PERSON` still leads, as it did for the biography pages, but the order below
it changes. `ORG`, `CARDINAL`, and `GPE` all outrank `DATE` here, while in
the chapter's corpus `DATE` was third behind `PERSON` and `ORG`. Reviews
count things (a rating out of ten, a number of stars, "one of the worst
films") far more than they cite calendar dates, which biography pages do
constantly for birth, death, and publication years.

5. Reconstruct subject-verb pairs the way the chapter does: select `VERB`
tokens as `verbs`, then join every `nsubj` token to the verb sitting at its
`head_idx` within the same `doc_id` and `sid`. Print the 10 most frequent
subject-verb pairs.

Nine of the ten pairs have `I` or `you` as the subject, the tenth is the
generic `it have`, and no movie-related noun makes the list. In the chapter's
biography pages, the most common subjects were the authors and their works
("author write", "work receive"). A review corpus's dependency structure is
dominated by its narrator instead of the thing being described, as you
would expect from a first-person genre.

6. Extract adjective-noun pairs the way the chapter does (`dep == "amod"`,
joined to the `NOUN` sitting at `head_idx`), restrict to pairs where the
noun is `"movie"` or `"film"`, and print the count for `"good"` and `"bad"`
by `label`.

Restricting to a grammatical attachment ("bad movie", not just "bad"
anywhere in the review) sharpens the signal for `bad` well beyond the last
notebook's whole-review counts: 16 negative uses of "bad movie"/"bad film"
against a single positive one, compared with 86 to 17 for `bad` anywhere.
`good` stays close to even, 9 negative against 8 positive. A tighter
grammatical filter helps when the word is doing real evaluative work, but it
does not handle negation: in "not a good movie", `good` still attaches to
`movie`, and the "not" hangs off a different word.

7. In the first block, find every negated predicate adjective: join tokens
with `dep == "neg"` to `ADJ` tokens with `dep in ["acomp", "attr"]` that
share the same `head_idx` within a `doc_id`/`sid` (this is the same "share a
head" logic question 5 used for subjects and verbs), and count how many
negated predicate adjectives exist in total. In the second block, count how
many of them have the negation word sitting in the position immediately
before the adjective, the only case a bigram search like question 1's
`shift(-1)` could ever catch.

Only 30 of the 76 negated predicate adjectives have the negation word
directly before them. The other 46 have something in between, sometimes a
single word ("never get tired") and sometimes several ("Never have I been so
frustrated"). A bigram search for "not X" would have missed 61% of these
negations, because English negation does not have to sit next to the word it
negates. The dependency join finds all 76 regardless of the gap, since it
follows a grammatical relationship instead of a distance. That is what the
last notebook's question 6 was missing: a way to tell "good" and "not good"
apart.

8. Run `DSText.kwic()` on the lemma `"acting"` with `window=5` and
`max_num=10`.

The display is not filtered by `label`, so the concordance shows both halves
of the corpus at once: "terrible", "decidedly bad", and "purely" sit near
"good acting" and "above average" within the same 10 lines. An
adjective-frequency table would not show this as directly. Here "acting" is
the noun being judged, and the words on either side of it are the ones
reviewers chose.

9. Compute the four complexity metrics from the chapter (mean sentence
length, type-token ratio, mean word length, and content-word ratio) per
document, join them into a single table, and print the mean of each metric
grouped by `label`.

None of the four metrics separates the two labels by much: sentence length
differs by about half a word, and word length by four hundredths of a
character. These are measures of *style*, and positive and negative reviews
of the same kind of movie, written by the same kind of person, have no
obvious reason to differ in style. As questions 6 and 7 showed, sentiment
lives in which words get chosen, not in how long or varied the sentences
are.

10. Count the number of `"!"` tokens per document, join that count onto
`docs`, and print the mean per-review count and the fraction of reviews
containing at least one `"!"`, grouped by `label`.

Here the two labels finally split: negative reviews average 1.21
exclamation points against 0.52 for positive ones, more than double. The
share of reviews with any exclamation point is much closer (36% against
31%), so the gap comes from a minority of negative reviews that use many. It
is a crude signal and says nothing about *what* is being exclaimed, but it
is a real difference in a place none of the chapter's four standard metrics
looks. The chapter does not cover punctuation, and the count needed nothing
beyond the `.filter()` and `.group_by()` chain from earlier chapters.